<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 14 · Reinforcement Learning Foundations

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
We turn RL foundations into code by defining a simple portfolio environment and solving it via tabular policy evaluation and simulation.

### Getting Help While Using RL
- **Appendix A** covers the math behind Bellman equations.
- **Chapter 8** provides the performance metrics used for rewards.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"

from dataclasses import dataclass

### Data Loading
We load the price panel once for this notebook.

In [ ]:
prices = pd.read_csv(DATA_PATH,
parse_dates=["Date"]).set_index("Date").sort_index().ffill()

## 1. Simple Portfolio Environment

We make the Markov Decision Process concrete by encoding portfolio state, actions, and rewards for a single‑asset rebalancing problem.

In [ ]:
@dataclass
class PortfolioEnv:
    prices: pd.DataFrame
    cost: float = 0.0005

    def reset(self):
        self.t = 0
        self.wealth = 1.0
        return self.state()

    def state(self):
        window = self.prices.iloc[max(0, self.t - 20): self.t]
        return window.pct_change().mean().fillna(0).values

    def step(self, action):  # action in {-1, 0, 1}
        ret = self.prices.iloc[self.t + 1]["AAPL"] / self.prices.iloc[self.t]["AAPL"] - 1
        reward = action * ret - self.cost * abs(action)
        self.wealth *= (1 + reward)
        self.t += 1
        done = self.t >= len(self.prices) - 2
        return self.state(), reward, done

env = PortfolioEnv(prices[["AAPL"]])
state0 = env.reset()
state0

## 2. Tabular Policy Evaluation Mock-Up

In this section we simulate random actions in the environment and estimate value via Monte Carlo, mirroring the Bellman intuition in code.

In [ ]:
%%time
actions = [-1, 0, 1]
policy = {a: 1 / len(actions) for a in actions}
value = 0.0
gamma = 0.95
n_episodes = 250

for episode in range(n_episodes):
    env.reset()
    done = False
    G = 0.0
    step = 0
    while not done:
        action = np.random.choice(actions)
        _, reward, done = env.step(action)
        G += (gamma ** step) * reward
        step += 1
    value += G
    if (episode + 1) % 50 == 0:
        avg_so_far = value / (episode + 1)
        print(f"Episode {episode + 1}/{n_episodes}: avg return {avg_so_far:.6f}")

value / n_episodes

## 3. Exercises
### Exercise 1 – Epsilon-Greedy Policy
Implement an epsilon-greedy action selection that biases toward the previous best action.
<details><summary>Hint</summary>
Track average rewards per action and mix exploitation with exploration.
</details>

### Exercise 2 – Reward Variants
Add a penalty for wealth volatility by subtracting <code>lambda * np.std(recent_returns)</code>.
<details><summary>Hint</summary>
Use a rolling window of returns inside the environment.
</details>

### Exercise 3 – Monte Carlo Evaluation
Run Monte Carlo episodes for a deterministic policy (e.g., always 1) and compare average wealth.
<details><summary>Hint</summary>
Call <code>env.step(1)</code> repeatedly and record <code>env.wealth</code> after each episode.
</details>


## 4. Takeaways for Chapter 14
- Even toy environments clarify how states/actions/rewards interact.
- Monte Carlo simulation provides intuition before moving to full Q-learning.
- Cost/penalty modeling ties RL formulations back to practical constraints.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">